# Nuer Whisper ASR Fine-tuning

Fine-tune Whisper on Nuer language data using Google Colab's free GPU.

**Setup:**
1. Upload your `nuer_whisper_asr` folder to Google Drive
2. Run all cells in order
3. Model will be saved to your Drive

## Step 1: Mount Google Drive
Connect your Drive where the data is stored.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Install Dependencies

In [3]:
!pip install -q transformers datasets accelerate torchaudio sentencepiece protobuf
!pip install -q evaluate jiwer  # For WER metric

print("✓ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.2 MB/s eta 0:00:00
✓ Dependencies installed


## Step 3: Set Paths

**IMPORTANT:** Update `DRIVE_PATH` to where you uploaded your project.

In [4]:
import os
from pathlib import Path

# UPDATE THIS to your project location in Drive
# Example: /content/drive/MyDrive/nuer_whisper_asr
DRIVE_PATH = Path("/content/drive/MyDrive/nuer_whisper_asr")

# Check if path exists
if not DRIVE_PATH.exists():
    print(f"❌ Path not found: {DRIVE_PATH}")
    print("Available files in Drive:")
    !ls -la /content/drive/MyDrive/ | head -20
else:
    print(f"✓ Project found at: {DRIVE_PATH}")
    print(f"  Data: {DRIVE_PATH / 'data'}")
    print(f"  Audio: {DRIVE_PATH / 'voice dataset'}")

✓ Project found at: /content/drive/MyDrive/nuer_whisper_asr
  Data: /content/drive/MyDrive/nuer_whisper_asr/data
  Audio: /content/drive/MyDrive/nuer_whisper_asr/voice dataset


## Step 4: Load and Verify Data

In [5]:
import json

# Load data
DATA_DIR = DRIVE_PATH / "data"
TRAIN_FILE = DATA_DIR / "train.json"
TEST_FILE = DATA_DIR / "test.json"

with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open(TEST_FILE, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

print(f"✓ Train samples: {len(train_data)}")
print(f"✓ Test samples: {len(test_data)}")

# Verify audio paths
AUDIO_DIR = DRIVE_PATH / "voice dataset"
sample_audio = AUDIO_DIR / train_data[0]['audio'].replace('../voice dataset/', '')

if sample_audio.exists():
    print(f"✓ Audio files accessible: {sample_audio.name}")
else:
    print(f"❌ Audio not found: {sample_audio}")

✓ Train samples: 150
✓ Test samples: 38
✓ Audio files accessible: ɛŋu_ci̱_bit_ɔ.wav


## Step 5: Prepare Dataset for HuggingFace

In [8]:
from datasets import Dataset, Audio
import torchaudio
import torch
import random
import numpy as np

def augment_audio(waveform, sample_rate=16000):
    """Apply audio augmentations to expand dataset."""
    # Volume change (±6dB)
    if random.random() < 0.5:
        gain_db = random.uniform(-6, 6)
        gain = 10 ** (gain_db / 20)
        waveform = waveform * gain

    # Time stretching (90-110% speed) via resampling
    if random.random() < 0.3:
        rate = random.uniform(0.9, 1.1)
        orig_len = waveform.shape[-1]
        new_len = int(orig_len / rate)
        waveform = torch.nn.functional.interpolate(
            waveform.unsqueeze(0), size=new_len, mode='linear', align_corners=False
        ).squeeze(0)
        # Pad or trim to original length
        if new_len < orig_len:
            waveform = torch.nn.functional.pad(waveform, (0, orig_len - new_len))
        else:
            waveform = waveform[:, :orig_len]

    return waveform

def prepare_dataset_with_augmentation(data, base_dir=AUDIO_DIR, augment_factor=3):
    """Prepare dataset with audio augmentations to expand small dataset."""
    audio_paths = []
    texts = []

    for item in data:
        rel_path = item["audio"].replace('../voice dataset/', '')
        audio_path = str(base_dir / rel_path)

        if os.path.exists(audio_path):
            # Add original
            audio_paths.append(audio_path)
            texts.append(item["text"])

            # Load and augment
            try:
                waveform, sr = torchaudio.load(audio_path)
                if sr != 16000:
                    resampler = torchaudio.transforms.Resample(sr, 16000)
                    waveform = resampler(waveform)

                # Create augmented versions
                aug_dir = base_dir / "augmented"
                aug_dir.mkdir(exist_ok=True)

                for aug_idx in range(augment_factor):
                    aug_waveform = augment_audio(waveform.clone(), 16000)
                    aug_name = f"aug{aug_idx}_{Path(audio_path).name}"
                    aug_path = aug_dir / aug_name

                    torchaudio.save(str(aug_path), aug_waveform, 16000)

                    audio_paths.append(str(aug_path))
                    texts.append(item["text"])
            except Exception as e:
                print(f"⚠ Skipping augmentation for {audio_path}: {e}")

    dataset = Dataset.from_dict({"audio": audio_paths, "text": texts})
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

    return dataset

print("Preparing datasets WITH AUGMENTATION (150 → ~600 samples)...")
train_dataset = prepare_dataset_with_augmentation(train_data, augment_factor=3)
test_dataset = prepare_dataset_with_augmentation(test_data, augment_factor=0)  # No augmentation for test

print(f"✓ Train: {len(train_dataset)} samples (augmented)")
print(f"✓ Test: {len(test_dataset)} samples (original)")

Preparing datasets WITH AUGMENTATION (150 → ~600 samples)...
✓ Train: 596 samples (augmented)
✓ Test: 38 samples (original)


## Step 6: Load Whisper Model

In [9]:
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration
)

MODEL_NAME = "openai/whisper-small"  # Options: tiny, base, small, medium

print(f"Loading {MODEL_NAME}...")

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_NAME, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Disable forced decoder IDs and suppress tokens for fine-tuning by setting them in generation_config
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

# Explicitly ensure these are NOT in model.config if they cause issues
# This addresses the ValueError: 'Some generation parameters are set in the model config.'
if hasattr(model.config, 'suppress_tokens'):
    del model.config.suppress_tokens
if hasattr(model.config, 'forced_decoder_ids'):
    del model.config.forced_decoder_ids

print(f"✓ Model loaded: {model.num_parameters()/1e6:.1f}M parameters")

Loading openai/whisper-small...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

✓ Model loaded: 241.7M parameters


## Step 7: Preprocess Dataset

In [10]:
def prepare_dataset_batch(batch):
    """Process a batch of data."""
    # Load audio
    audio = batch["audio"]

    # Compute log-Mel input features
    input_features = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Encode target text
    labels = tokenizer(batch["text"]).input_ids

    # Return only the new columns
    return {
        "input_features": input_features,
        "labels": labels
    }

print("Processing train dataset...")
# Pass an explicit list of columns to remove. The function now returns only new columns.
train_dataset = train_dataset.map(prepare_dataset_batch, remove_columns=['audio', 'text'])

print("Processing test dataset...")
# Pass an explicit list of columns to remove.
test_dataset = test_dataset.map(prepare_dataset_batch, remove_columns=['audio', 'text'])

print("✓ Preprocessing complete")

Processing train dataset...


Map:   0%|          | 0/596 [00:00<?, ? examples/s]

Processing test dataset...


Map:   0%|          | 0/38 [00:00<?, ? examples/s]

✓ Preprocessing complete


## Step 8: Setup Data Collator

In [11]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("✓ Data collator ready")

✓ Data collator ready


## Step 9: Define WER Metric

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad token id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("✓ WER metric ready")

✓ WER metric ready


## Step 10: Configure Training

In [ ]:
from transformers import Seq2SeqTrainingArguments, EarlyStoppingCallback
import transformers
import os
from pathlib import Path

# Create output directory in Drive
OUTPUT_DIR = DRIVE_PATH / "models" / "whisper-nuer-improved"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Transformers version: {transformers.__version__}")

# IMPROVED CONFIGURATION for low-resource Nuer ASR
# Key changes: higher LR (5e-5), more steps (4000), early stopping

NUM_TRAIN_SAMPLES = 150
MAX_STEPS = 4000          # Increased from 500 - need more steps to override English bias
LEARNING_RATE = 5e-5      # Increased from 1e-5 - higher LR to overcome pretrained bias
WARMUP_STEPS = 200        # Gradual warmup for stability

try:
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        max_steps=MAX_STEPS,
        lr_scheduler_type="cosine",
        gradient_checkpointing=True,
        fp16=True,
        evaluation_strategy="steps",
        per_device_eval_batch_size=8,
        predict_with_generate=True,
        generation_max_length=128,
        save_steps=400,
        eval_steps=400,
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        push_to_hub=False,
        report_to=["tensorboard"],
        save_total_limit=3,
    )
    print(f"✓ Training config ready")
    print(f"  Learning rate: {LEARNING_RATE} (higher to override English bias)")
    print(f"  Max steps: {MAX_STEPS} (more training needed)")
    print(f"  Early stopping: Enabled (patience=5)")

except TypeError as e:
    if "evaluation_strategy" in str(e):
        print("⚠ Using simplified training config...")
        training_args = Seq2SeqTrainingArguments(
            output_dir=str(OUTPUT_DIR),
            per_device_train_batch_size=8,
            gradient_accumulation_steps=2,
            learning_rate=LEARNING_RATE,
            warmup_steps=WARMUP_STEPS,
            max_steps=MAX_STEPS,
            lr_scheduler_type="cosine",
            gradient_checkpointing=True,
            fp16=True,
            per_device_eval_batch_size=8,
            predict_with_generate=True,
            generation_max_length=128,
            save_steps=400,
            logging_steps=100,
            save_total_limit=3,
            report_to=["tensorboard"],
            push_to_hub=False,
        )
        print(f"✓ Training config ready (simplified)")
    else:
        raise

print(f"  Output: {OUTPUT_DIR}")
print(f"  Steps: {training_args.max_steps}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")

Transformers version: 5.0.0
❌ 'evaluation_strategy' not recognized. Likely an older transformers version.
  Attempting to configure training arguments without specific evaluation strategy.
✓ Training config ready (adjusted for compatibility).
  Output: /content/drive/MyDrive/nuer_whisper_asr/models/whisper-nuer
  Steps: 500
  Batch size: 8


## Step 11: Start Training 🚀

In [ ]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]  # Stop if WER doesn't improve for 5 evals
)

print("\n" + "="*60)
print("Starting training with improved configuration...")
print("Key improvements:")
print(f"  - Learning rate: 5e-5 (was 1e-5) - overrides English bias faster")
print(f"  - Max steps: 4000 (was 500) - trains much longer")
print(f"  - Early stopping: patience=5 - prevents overfitting")
print(f"  - Cosine LR schedule: smoother decay")
print("="*60 + "\n")

trainer.train()

print("\n" + "="*60)
print("Training complete!")
print("="*60)


Starting training...



Step,Training Loss
25,6.920353
50,1.608193
75,0.257852
100,0.072877
125,0.027489
150,0.008628
175,0.010112
200,0.005471
225,0.002742
250,0.004150


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Step 12: Save Final Model

In [ ]:
FINAL_MODEL_DIR = DRIVE_PATH / "models" / "whisper-nuer-final"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Saving model...")
trainer.save_model(str(FINAL_MODEL_DIR))
processor.save_pretrained(str(FINAL_MODEL_DIR))

print(f"✓ Model saved to: {FINAL_MODEL_DIR}")
print(f"\nModel size: {sum(f.stat().st_size for f in FINAL_MODEL_DIR.rglob('*') if f.is_file()) / 1e6:.1f} MB")

## Step 13: Test the Model

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T

# CRITICAL FIX: Function to transcribe with forced decoder IDs
# This prevents English translation and forces Nuer transcription
def transcribe_nuer_fixed(model, processor, audio_path, device="cuda"):
    """
    Transcribe with proper decoder forcing for Nuer transcription.
    This prevents English output by explicitly setting <|transcribe|> task token.
    """
    # Load audio
    waveform, sample_rate = torchaudio.load(str(audio_path))

    # Resample to 16kHz if necessary
    TARGET_SAMPLING_RATE = 16000
    if sample_rate != TARGET_SAMPLING_RATE:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=TARGET_SAMPLING_RATE)
        waveform = resampler(waveform)
        sample_rate = TARGET_SAMPLING_RATE

    # Process audio
    input_features = processor(
        waveform.squeeze().numpy(),
        sampling_rate=sample_rate,
        return_tensors="pt"
    ).input_features.to(device)

    # CRITICAL: Get special token IDs for decoder forcing
    tokenizer = processor.tokenizer
    transcribe_token_id = tokenizer.convert_tokens_to_ids("<|transcribe|>")
    notimestamps_token_id = tokenizer.convert_tokens_to_ids("<|notimestamps|>")

    # Build forced_decoder_ids to tell Whisper:
    # "This is a TRANSCRIPTION task (not translation), no timestamps"
    forced_decoder_ids = [
        (1, transcribe_token_id),      # Position 1: task = transcribe
        (2, notimestamps_token_id),    # Position 2: no timestamps
    ]

    # Generate with optimized settings
    model.eval()
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=128,           # Limit output length
            temperature=0.0,              # Greedy decoding (no randomness)
            no_repeat_ngram_size=3,       # Prevent repetition loops
            do_sample=False,              # No sampling
            num_beams=1,                  # Greedy (not beam search)
            forced_decoder_ids=forced_decoder_ids,  # CRITICAL: Force transcription mode
            suppress_tokens=[],           # Don't suppress any tokens (allow Nuer)
        )

    # Decode
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription


# Test on multiple samples
print("="*60)
print("Testing with FORCED DECODER IDs (fixes English output)")
print("="*60)

model = model.to("cuda")
model.eval()

# Test on first 5 test samples
for i in range(min(5, len(test_data))):
    sample = test_data[i]
    audio_path = AUDIO_DIR / sample['audio'].replace('../voice dataset/', '')

    prediction = transcribe_nuer_fixed(model, processor, audio_path)

    print(f"\nSample {i+1}:")
    print(f"  Ground truth: {sample['text']}")
    print(f"  Prediction:   {prediction}")
    match = prediction.strip().lower() == sample['text'].strip().lower()
    print(f"  Match: {'✓' if match else '✗'}")

print("\n" + "="*60)
print("If still getting English output, you need to RETRAIN with:")
print("  - Higher learning rate (5e-5 instead of 1e-5)")
print("  - More steps (4000 instead of 500)")
print("  - Data augmentation (expand 150 → 600 samples)")
print("="*60)

Testing on: ciek
Audio: ciek.wav

Ground truth: ciek
Prediction:    Hit


## Next Steps

1. **Download model** (if needed locally):
   - Find it in your Google Drive: `models/whisper-nuer-final/`

2. **Use for inference**:
   - Load with: `pipeline('automatic-speech-recognition', model=path)`

3. **Quantize for mobile** (future step):
   - Convert to ONNX or TFLite for edge deployment